# 🛩️ UAV Degradation Predictive Model — Training Notebook

**Architecture**: Conv1D → BiGRU × 2 → MultiHead Self-Attention → Dense  
**Features**: 59 engineered features from 13 sensor datasets  
**Classes**: 5 (Nominal / Vibration / Sensor Drift / GPS Fault / Battery Fault)  
**Backend**: PyTorch + GPU (T4)

---

### Instructions
1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells in order
3. Upload dataset ZIP when prompted
4. Download the trained model files at the end

## 1. Setup & GPU Check

In [ ]:
import torch
import numpy as np
import pandas as pd
import os, pickle, time, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected — training will be slower. Go to Runtime → Change runtime type → GPU')

## 2. Upload Dataset

**Before running the next cell**, zip your entire `dataset/` folder:
```
dataset/
  ATT/ALL_FAIL_LOG_ATT.csv
  BARO/ALL_FAIL_LOG_BARO.csv
  BAT/ALL_FAIL_LOG_BAT_0.csv
  CTUN/ALL_FAIL_LOG_CTUN.csv
  GPS/ALL_FAIL_LOG_GPS_0.csv
  IMU/ALL_FAIL_LOG_IMU_0_Random.csv
  MAG/ALL_FAIL_LOG_MAG_0.csv
  MOTB/ALL_FAIL_LOG_MOTB.csv
  PSCD/ALL_FAIL_LOG_PSCD.csv
  RATE/ALL_FAIL_LOG_RATE.csv
  VIBE/ALL_FAIL_LOG_VIBE_0_Random.csv
  XKF1/ALL_FAIL_LOG_XKF1_0_Random.csv
  Fusion_Data.csv
```

Name the zip file **`dataset.zip`** and upload it when prompted.

In [ ]:
from google.colab import files
import zipfile

print('📁 Upload dataset.zip (your zipped dataset/ folder)...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\nReceived: {zip_name} ({len(uploaded[zip_name])/1e6:.1f} MB)')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
    print(f'Extracted {len(z.namelist())} files')

# Auto-detect if files are in dataset/ or at root
if os.path.isdir('dataset'):
    BASE_DATA = 'dataset'
elif os.path.isdir('ATT'):
    BASE_DATA = '.'
else:
    # search for ATT folder
    for root, dirs, _files in os.walk('.'):
        if 'ATT' in dirs:
            BASE_DATA = root
            break
    else:
        raise FileNotFoundError('Could not find ATT/ folder. Check your zip structure.')

print(f'Dataset root: {BASE_DATA}/')
print('Contents:', os.listdir(BASE_DATA))

## 3. Load All 13 Datasets

In [ ]:
def load_csv(rel_path, dedup_cols=False):
    full = os.path.join(BASE_DATA, rel_path)
    if not os.path.exists(full):
        print(f'  [WARN] Not found: {full}')
        return pd.DataFrame()
    df = pd.read_csv(full, low_memory=False)
    if dedup_cols:
        df = df.loc[:, ~df.columns.duplicated()]
    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0.0, inplace=True)
    print(f'  {os.path.basename(rel_path):42s}  {len(df):6d} rows  cols={list(df.columns[:5])}...')
    return df

print('Loading 12 sensor datasets + fusion data...\n')
att  = load_csv('ATT/ALL_FAIL_LOG_ATT.csv')
baro = load_csv('BARO/ALL_FAIL_LOG_BARO.csv')
bat  = load_csv('BAT/ALL_FAIL_LOG_BAT_0.csv')
ctun = load_csv('CTUN/ALL_FAIL_LOG_CTUN.csv')
gps  = load_csv('GPS/ALL_FAIL_LOG_GPS_0.csv')
imu  = load_csv('IMU/ALL_FAIL_LOG_IMU_0_Random.csv')
mag  = load_csv('MAG/ALL_FAIL_LOG_MAG_0.csv')
motb = load_csv('MOTB/ALL_FAIL_LOG_MOTB.csv')
pscd = load_csv('PSCD/ALL_FAIL_LOG_PSCD.csv')
rate = load_csv('RATE/ALL_FAIL_LOG_RATE.csv')
vibe = load_csv('VIBE/ALL_FAIL_LOG_VIBE_0_Random.csv', dedup_cols=True)
xkf1 = load_csv('XKF1/ALL_FAIL_LOG_XKF1_0_Random.csv')

print(f'\n✓ All datasets loaded. Master timeline: ATT with {len(att)} rows')

## 4. Build 59-Feature Matrix

| Slot | Dataset | Features | Count |
|------|---------|----------|-------|
| 0-7 | ATT | DesRoll Roll DesPitch Pitch DesYaw Yaw ErrRP ErrYaw | 8 |
| 8-14 | BAT | Volt VoltR Curr CurrTot Temp Res RemPct | 7 |
| 15-17 | BARO | Alt Press CRt | 3 |
| 18-20 | GPS | NSats HDop Spd | 3 |
| 21-27 | IMU | GyrX GyrY GyrZ AccX AccY AccZ Temp | 7 |
| 28-31 | MAG | MagX MagY MagZ Health | 4 |
| 32-35 | MOTB | LiftMax BatVolt BatRes ThLimit | 4 |
| 36-39 | PSCD | TPD PD DVD VD | 4 |
| 40-47 | RATE | RDes R PDes P YDes Y ADes A | 8 |
| 48-51 | VIBE | VibeX VibeY VibeZ Clip | 4 |
| 52-57 | XKF1 | VN VE VD PN PE OH | 6 |
| 58 | CTUN | ThO | 1 |
| | | **Total** | **59** |

In [ ]:
N_FEATURES = 59
SEQ_LEN    = 40
N_CLASSES  = 5
SEED       = 42
N          = len(att)
LABEL_COL  = 'lables'   # ATT dataset has this typo

np.random.seed(SEED)
torch.manual_seed(SEED)

def get_val(df, idx, col, default=0.0):
    if df.empty or col not in df.columns:
        return default
    mapped = int((idx / N) * len(df)) % len(df)
    try:
        v = float(df.iloc[mapped].get(col, default))
        return v if np.isfinite(v) else default
    except:
        return default

print(f'Building {N} × {N_FEATURES} feature matrix...\n')
X_list, y_list = [], []

for i in range(N):
    g = lambda df, col, d=0.0: get_val(df, i, col, d)
    row = [
        # ATT (8)
        g(att,'DesRoll'), g(att,'Roll'), g(att,'DesPitch'), g(att,'Pitch'),
        g(att,'DesYaw'),  g(att,'Yaw'),  g(att,'ErrRP'),    g(att,'ErrYaw'),
        # BAT (7)
        g(bat,'Volt'),   g(bat,'VoltR'),  g(bat,'Curr'),    g(bat,'CurrTot'),
        g(bat,'Temp'),   g(bat,'Res'),    g(bat,'RemPct'),
        # BARO (3)
        g(baro,'Alt'),   g(baro,'Press'), g(baro,'CRt'),
        # GPS (3)
        g(gps,'NSats'),  g(gps,'HDop'),   g(gps,'Spd'),
        # IMU (7)
        g(imu,'abGyrX'), g(imu,'abGyrY'), g(imu,'abGyrZ'),
        g(imu,'abAccX'), g(imu,'abAccY'), g(imu,'abAccZ'), g(imu,'abT'),
        # MAG (4)
        g(mag,'MagX'),   g(mag,'MagY'),   g(mag,'MagZ'),   g(mag,'Health'),
        # MOTB (4)
        g(motb,'LiftMax'), g(motb,'BatVolt'), g(motb,'BatRes'), g(motb,'ThLimit'),
        # PSCD (4)
        g(pscd,'TPD'),   g(pscd,'PD'),    g(pscd,'DVD'),   g(pscd,'VD'),
        # RATE (8)
        g(rate,'RDes'),  g(rate,'R'),     g(rate,'PDes'),  g(rate,'P'),
        g(rate,'YDes'),  g(rate,'Y'),     g(rate,'ADes'),  g(rate,'A'),
        # VIBE (4)
        g(vibe,'VibeX'), g(vibe,'VibeY'), g(vibe,'VibeZ'), g(vibe,'Clip'),
        # XKF1 (6)
        g(xkf1,'VN'),    g(xkf1,'VE'),   g(xkf1,'VD'),
        g(xkf1,'PN'),    g(xkf1,'PE'),   g(xkf1,'OH'),
        # CTUN (1)
        g(ctun,'ThO'),
    ]
    assert len(row) == N_FEATURES, f'Feature mismatch: {len(row)}'
    X_list.append(row)
    y_list.append(int(att.iloc[i].get(LABEL_COL, 0)))

    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{N} rows ...')

X_raw = np.array(X_list, dtype=np.float32)
y_raw = np.array(y_list, dtype=np.int64)

cls_names = ['Nominal', 'Vibration', 'Sensor Drift', 'GPS Fault', 'Battery Fault']
print(f'\nFeature matrix: {X_raw.shape}')
for cls, cnt in zip(*np.unique(y_raw, return_counts=True)):
    pct = cnt / len(y_raw) * 100
    bar = '█' * int(pct / 2)
    print(f'  {cls} {cls_names[cls]:15s}: {cnt:5d}  ({pct:5.1f}%) {bar}')

## 5. Scale + Create Sequences

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# RobustScaler handles outliers in RATE/PSCD much better than MinMaxScaler
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = np.clip(X_scaled, -5.0, 5.0).astype(np.float32)

print(f'Scaled range: [{X_scaled.min():.2f}, {X_scaled.max():.2f}]')

# Sliding window sequences
def make_sequences(X, y, seq_len):
    Xs = np.lib.stride_tricks.sliding_window_view(X, (seq_len, X.shape[1]))[::1, 0]
    ys = y[seq_len:]
    return Xs.astype(np.float32), ys.astype(np.int64)

X_seq, y_seq = make_sequences(X_scaled, y_raw, SEQ_LEN)
print(f'Sequences: {X_seq.shape}  Labels: {y_seq.shape}')

# Split: 70% train / 15% val / 15% test
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_seq, y_seq, test_size=0.15, random_state=SEED, stratify=y_seq)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.176, random_state=SEED, stratify=y_tmp)  # 0.176 * 0.85 ≈ 0.15

print(f'\nTrain: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')

# Class weights for imbalanced data
cw_arr = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_train)
cw_tensor = torch.tensor(cw_arr, dtype=torch.float32).to(DEVICE)
print(f'Class weights: { {cls_names[i]: round(float(cw_arr[i]),2) for i in range(N_CLASSES)} }')

## 6. Model Architecture

```
Input (40, 59)
 ↓
Conv1D(128, k=5) → BatchNorm → GELU
Conv1D(64,  k=3) → BatchNorm → GELU → Dropout(0.2)
 ↓
BiGRU(128) → Dropout(0.3)
BiGRU(64)  → Dropout(0.25)
 ↓
MultiheadAttention(4 heads, k=32) + Residual + LayerNorm
 ↓
GlobalAveragePool → Dense(128) → Dense(64) → Dense(5)
```

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class UAVFaultNet(nn.Module):
    """
    Conv1D → BiGRU × 2 → MultiheadAttention → GlobalAvgPool → Dense
    """
    def __init__(self, n_feat=59, seq_len=40, n_cls=5):
        super().__init__()

        # Temporal convolutional front-end
        self.conv = nn.Sequential(
            nn.Conv1d(n_feat, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.GELU(),
            nn.Dropout(0.2),
        )

        # Bidirectional GRU stack
        self.gru1  = nn.GRU(64,  128, bidirectional=True, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.gru2  = nn.GRU(256,  64, bidirectional=True, batch_first=True)
        self.drop2 = nn.Dropout(0.25)

        # Multi-head self-attention
        self.attn  = nn.MultiheadAttention(128, num_heads=4, dropout=0.1, batch_first=True)
        self.norm  = nn.LayerNorm(128)
        self.drop3 = nn.Dropout(0.15)

        # Classification head
        self.head = nn.Sequential(
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, n_cls),
        )

    def forward(self, x):
        # x: [B, L, F]
        xc = self.conv(x.transpose(1, 2)).transpose(1, 2)  # [B, L, 64]
        xg, _ = self.gru1(xc);   xg = self.drop1(xg)      # [B, L, 256]
        xg, _ = self.gru2(xg);   xg = self.drop2(xg)      # [B, L, 128]
        xa, _ = self.attn(xg, xg, xg)                      # [B, L, 128]
        xa = self.norm(xg + self.drop3(xa))                 # residual + LN
        xp = xa.mean(dim=1)                                 # [B, 128]
        return self.head(xp)                                # [B, n_cls]


model = UAVFaultNet(N_FEATURES, SEQ_LEN, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'UAVFaultNet — {n_params:,} trainable parameters')
print(f'Device: {DEVICE}')
print(model)

## 7. Train

In [ ]:
# DataLoaders
def to_loader(X, y, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

train_ld = to_loader(X_train, y_train, batch_size=64, shuffle=True)
val_ld   = to_loader(X_val,   y_val,   batch_size=128, shuffle=False)
test_ld  = to_loader(X_test,  y_test,  batch_size=128, shuffle=False)

# Loss, Optimizer, Scheduler
criterion = nn.CrossEntropyLoss(weight=cw_tensor)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# Training config
MAX_EPOCHS = 80
PATIENCE   = 15
best_val_acc, patience_ctr, best_state = 0.0, 0, None
history = []

print(f'Starting training: {MAX_EPOCHS} epochs, batch_size=64, patience={PATIENCE}')
print(f'{"Ep":>4s}  {"Train Loss":>10s}  {"Train Acc":>9s}  {"Val Loss":>10s}  {"Val Acc":>9s}  {"LR":>9s}  {"Time":>6s}')
print('-' * 72)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    # === Train ===
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for Xb, yb in train_ld:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step(epoch - 1 + t_total / len(train_ld.dataset))
        t_loss    += loss.item() * len(yb)
        t_correct += (logits.argmax(1) == yb).sum().item()
        t_total   += len(yb)

    # === Validate ===
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in val_ld:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            v_loss    += criterion(logits, yb).item() * len(yb)
            v_correct += (logits.argmax(1) == yb).sum().item()
            v_total   += len(yb)

    tl = t_loss / t_total
    ta = t_correct / t_total * 100
    vl = v_loss / v_total
    va = v_correct / v_total * 100
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'train_loss': tl, 'train_acc': ta,
                    'val_loss': vl, 'val_acc': va, 'lr': lr})

    marker = ''
    if va > best_val_acc:
        best_val_acc = va
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
        marker = ' ✓ BEST'
    else:
        patience_ctr += 1

    print(f'{epoch:4d}  {tl:10.4f}  {ta:8.2f}%  {vl:10.4f}  {va:8.2f}%  {lr:9.6f}  {elapsed:5.1f}s{marker}')

    if patience_ctr >= PATIENCE:
        print(f'\n⏹ Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

print(f'\n✓ Training complete! Best val_acc = {best_val_acc:.2f}%')

## 8. Evaluate on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(best_state)
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for Xb, yb in test_ld:
        logits = model(Xb.to(DEVICE))
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(yb.tolist())
        all_probs.extend(probs.tolist())

from sklearn.metrics import accuracy_score
test_acc = accuracy_score(all_labels, all_preds) * 100

print('=' * 55)
print(f'  TEST ACCURACY : {test_acc:.2f}%')
print(f'  BEST VAL ACC  : {best_val_acc:.2f}%')
print('=' * 55)
print()
print(classification_report(all_labels, all_preds, target_names=cls_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print('Confusion Matrix:')
print(f'{"":>18s}', end='')
for n in cls_names:
    print(f'{n[:8]:>10s}', end='')
print()
for i, row in enumerate(cm):
    print(f'{cls_names[i]:>18s}', end='')
    for v in row:
        print(f'{v:10d}', end='')
    print()

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs, [h['train_loss'] for h in history], label='Train', color='#3b82f6')
axes[0].plot(epochs, [h['val_loss'] for h in history],   label='Val',   color='#ef4444')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [h['train_acc'] for h in history], label='Train', color='#3b82f6')
axes[1].plot(epochs, [h['val_acc'] for h in history],   label='Val',   color='#10b981')
axes[1].axhline(y=best_val_acc, color='#f59e0b', linestyle='--', alpha=0.7, label=f'Best: {best_val_acc:.1f}%')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy %')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(epochs, [h['lr'] for h in history], color='#a855f7')
axes[2].set_title('Learning Rate', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 10. Confusion Matrix Heatmap

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cls_names, yticklabels=cls_names, ax=ax)
ax.set_title(f'Confusion Matrix — Test Acc: {test_acc:.2f}%', fontweight='bold')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save & Download Model Files

This saves 3 files you need to copy back to your project:
- `uav_model.pt` — trained PyTorch model
- `scaler.pkl` — RobustScaler (59 features)
- `model_meta.pkl` — metadata for ml_api.py

In [ ]:
# Save model
torch.save({
    'model_state_dict': best_state,
    'model_class':      'UAVFaultNet',
    'n_features':       N_FEATURES,
    'seq_len':          SEQ_LEN,
    'n_classes':        N_CLASSES,
    'best_val_acc':     best_val_acc,
    'test_acc':         test_acc,
    'cls_names':        cls_names,
    'history':          history,
}, 'uav_model.pt')

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save meta
meta = {
    'n_features':  N_FEATURES,
    'seq_len':     SEQ_LEN,
    'n_classes':   N_CLASSES,
    'scaler_type': 'RobustScaler',
    'backend':     'pytorch',
}
with open('model_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

# Save training log
pd.DataFrame(history).to_csv('training_log.csv', index=False)

print('Saved files:')
for f in ['uav_model.pt', 'scaler.pkl', 'model_meta.pkl', 'training_log.csv']:
    sz = os.path.getsize(f) / 1e6
    print(f'  ✓ {f:25s}  {sz:.2f} MB')

In [ ]:
# Download all model files as a single zip
import shutil

zip_name = 'uav_trained_model'
os.makedirs(zip_name, exist_ok=True)
for f in ['uav_model.pt', 'scaler.pkl', 'model_meta.pkl', 'training_log.csv',
          'training_curves.png', 'confusion_matrix.png']:
    if os.path.exists(f):
        shutil.copy(f, zip_name)

shutil.make_archive(zip_name, 'zip', zip_name)
print(f'📦 Created: {zip_name}.zip ({os.path.getsize(zip_name + ".zip") / 1e6:.2f} MB)')
print()
print('Downloading...')

from google.colab import files
files.download(f'{zip_name}.zip')

## 12. After Download — What To Do

1. **Unzip** `uav_trained_model.zip`
2. **Copy** these 3 files to your project root:
   ```
   Degradation_Predictive_System/
     uav_model.pt        ← trained model
     scaler.pkl           ← RobustScaler
     model_meta.pkl       ← metadata
   ```
3. **Run** `start_system.bat` — the API will automatically load the new model

That's it! 🎉